# Task 3: Feature Engineering Analysis

**Mục tiêu:** Design engagement score formula và filter thresholds.

**Output:** `results/feature_engineering.json`

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw_tiktok_data.csv', encoding='utf-8-sig')
print(f"✅ Loaded {len(df)} videos")

## 2. Extract Engagement Metrics

In [ ]:
# TODO: Extract metrics từ CSV
# Column names có thể là: playCount, diggCount, commentCount, shareCount

# Hoặc rename cho dễ:
df_metrics = df[[
    'id',
    'text',
    'playCount',      # views
    'diggCount',      # likes
    'commentCount',   # comments
    'shareCount'      # shares
]].copy()

df_metrics.rename(columns={
    'playCount': 'views',
    'diggCount': 'likes',
    'commentCount': 'comments',
    'shareCount': 'shares'
}, inplace=True)

# Remove nulls
df_metrics = df_metrics.dropna(subset=['views', 'likes', 'comments', 'shares'])

print(f"✅ Cleaned data: {len(df_metrics)} videos")
df_metrics.head()

## 3. Correlation Analysis

In [ ]:
# TODO: Tính correlation matrix
# Hint: df_metrics[['views', 'likes', 'comments', 'shares']].corr()

corr_matrix = None  # TODO

print("Correlation Matrix:")
print(corr_matrix)

In [ ]:
# TODO: Vẽ heatmap
# Hint: sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')

plt.figure(figsize=(8, 6))
# TODO: Heatmap
plt.title('Correlation Matrix - Engagement Metrics')
plt.show()

## 4. Design Engagement Score

Formula: `engagement = likes*w1 + comments*w2 + shares*w3 + views*w4`

**Reasoning:**
- Comments/Shares valuable hơn Likes (người dùng phải effort hơn)
- Views rất nhiều nên weight nhỏ

**Đề xuất weights:**
- likes: 1.0
- comments: 2.0 (valuable hơn likes)
- shares: 3.0 (valuable nhất)
- views: 0.01 (scale down vì số lớn)

**Mày có thể tune weights khác dựa trên correlation!**

In [ ]:
# TODO: Thử nhiều weight combinations

# Version 1: Baseline
w_likes_v1 = 1.0
w_comments_v1 = 2.0
w_shares_v1 = 3.0
w_views_v1 = 0.01

df_metrics['engagement_v1'] = (
    df_metrics['likes'] * w_likes_v1 +
    df_metrics['comments'] * w_comments_v1 +
    df_metrics['shares'] * w_shares_v1 +
    df_metrics['views'] * w_views_v1
)

# TODO: Thử version 2, 3 với weights khác
# Ví dụ:
# w_likes_v2 = 1.0
# w_comments_v2 = 3.0
# ...

print("Engagement score (v1) distribution:")
print(df_metrics['engagement_v1'].describe())

In [ ]:
# TODO: Visualize engagement distribution
df_metrics['engagement_v1'].hist(bins=50)
plt.title('Engagement Score Distribution')
plt.xlabel('Engagement Score')
plt.show()

## 5. Filter Thresholds

Đề xuất thresholds để filter low-quality videos.

In [ ]:
# TODO: Analyze distribution để chọn thresholds

# Example: Videos với views < 1000 chiếm bao nhiêu %?
low_views_count = (df_metrics['views'] < 1000).sum()
low_views_pct = low_views_count / len(df_metrics) * 100

print(f"Videos with views < 1000: {low_views_count} ({low_views_pct:.1f}%)")

# TODO: Tương tự cho likes, engagement

In [ ]:
# TODO: Đề xuất thresholds

min_views = 1000  # TODO: Điều chỉnh
min_likes = 50    # TODO: Điều chỉnh
min_engagement = 100  # TODO: Điều chỉnh

# Test: Bao nhiêu % videos sẽ được giữ lại?
filtered = df_metrics[
    (df_metrics['views'] >= min_views) &
    (df_metrics['likes'] >= min_likes) &
    (df_metrics['engagement_v1'] >= min_engagement)
]

print(f"\nAfter filtering:")
print(f"  Kept: {len(filtered)} videos ({len(filtered)/len(df_metrics)*100:.1f}%)")
print(f"  Removed: {len(df_metrics) - len(filtered)} videos")

## 6. Text Combination Strategy

In [ ]:
# TODO: Test text combination strategies

# Option A: Chỉ text
# Option B: text + hashtags
# Option C: text + hashtags + author

# Ví dụ với 1 video:
sample_video = df.iloc[0]

text = sample_video['text']

# Extract hashtags (từ hashtags/0, hashtags/1, ...)
hashtag_cols = [col for col in df.columns if col.startswith('hashtags/')]
hashtags = [sample_video[col] for col in hashtag_cols if pd.notna(sample_video[col])]

print("Option A (text only):")
print(text)
print()

print("Option B (text + hashtags):")
combined_text = f"{text} {' '.join(hashtags)}"
print(combined_text)
print()

# TODO: Đề xuất strategy nào tốt nhất? Tại sao?

## 7. Save Results

In [ ]:
# TODO: Fill in kết quả analysis

results = {
    "engagement_formula": {
        "likes_weight": w_likes_v1,
        "comments_weight": w_comments_v1,
        "shares_weight": w_shares_v1,
        "views_weight": w_views_v1,
        "formula_string": f"likes*{w_likes_v1} + comments*{w_comments_v1} + shares*{w_shares_v1} + views*{w_views_v1}"
    },
    "filter_thresholds": {
        "min_views": min_views,
        "min_likes": min_likes,
        "min_engagement_score": min_engagement,
        "reasoning": f"TODO: Giải thích tại sao chọn thresholds này"
    },
    "text_combination": {
        "strategy": "text + hashtags",  # TODO: Chọn strategy
        "reasoning": "TODO: Giải thích tại sao"
    },
    "correlation_matrix": {
        "views_likes": 0.0,  # TODO: Fill từ corr_matrix
        "views_comments": 0.0,  # TODO
        "likes_comments": 0.0  # TODO
    }
}

# Save JSON
output_path = Path('../results/feature_engineering.json')
output_path.parent.mkdir(exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"✅ Results saved to {output_path}")

In [ ]:
# Preview
print(json.dumps(results, indent=2, ensure_ascii=False))

---

## ✅ DONE!

Kiểm tra file `results/feature_engineering.json` đã được tạo chưa.